#### Este notebook es un clon del 07-result_country_prod_company_managed_table de la carpeta transformation con algunas cosas modificadas para que sea carga incremental, las cosas modificadas tienen el comentario al lado, básicamente lo que se modifica es:
1. Se agrega el includes de common_functions
2. Se agrega el widget v_file_date con el nombre de las carpetas de carga incremental del contenedor bronze-inc
3. Se agrega filtro para leer las managed table particionadas de silver-inc (las que tienen datos que cambian cada semana)
4. Se modifica el mapeo de created_date, que antes mapeaba current_timestamp, ahora mapea el file_date
5. Se cambia la forma de crear la managed table en el ultimo paso, el esquema, el modo de escritura, la particion, y además se agrega un paso previo que elimine registros en caso de duplicados

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")






In [0]:
movie_df = spark.read.table("movie_silver_inc.movies") \
                     .filter(f"file_date = '{v_file_date}'") \
                     .filter("year_release_date > 2010")
                     
                     

In [0]:
country_df = spark.read.table("movie_silver_inc.countries")

In [0]:
movie_company_df = spark.read.table("movie_silver_inc.movies_companies") \
                             .filter(f"file_date = '{v_file_date}'")

In [0]:
production_country_df = spark.read.table("movie_silver_inc.productions_countries") \
                                  .filter(f"file_date = '{v_file_date}'")

In [0]:
production_company_df = spark.read.table("movie_silver_inc.production_companies") \
                                  .filter(f"file_date = '{v_file_date}'")

#### Join "production_countries" y "countries"

In [0]:
countries_prod_coun_df = production_country_df.join(country_df,
                                    production_country_df.country_id == country_df.country_id,
                                    "inner") \
                                    .select(country_df.country_name, country_df.country_id, production_country_df.movie_id)


#### Join "movie_companies" y "production_companies"

In [0]:
companies_prod_comp_df = movie_company_df.join(production_company_df,
                                    movie_company_df.company_id == production_company_df.company_id,
                                    "inner") \
                                    .select(production_company_df.company_name, production_company_df.company_id, movie_company_df.movie_id)

#### Join "movie_df", "countries_prod_coun_df" y "companies_prod_comp_df"

In [0]:
results_country_prod_company_df = movie_df.join(countries_prod_coun_df,
                                                movie_df.movie_id == countries_prod_coun_df.movie_id,
                                                "inner") \
                                            .join(companies_prod_comp_df,
                                                  movie_df.movie_id == companies_prod_comp_df.movie_id,
                                                  "inner") \
                                            .select(movie_df.movie_id, movie_df.title, movie_df.budget, movie_df.revenue, movie_df.duration_time, 
                                                    movie_df.release_date, countries_prod_coun_df.country_id, countries_prod_coun_df.country_name, companies_prod_comp_df.company_id, companies_prod_comp_df.company_name)
                                            

In [0]:
from pyspark.sql.functions import lit






In [0]:
results_df = results_country_prod_company_df.withColumn("created_date", lit(v_file_date))


##### Ordenar por la columna "title" de manera ascendente

In [0]:
results_order_by_df = results_df.orderBy(results_df.title.asc())

In [0]:
display(results_order_by_df)

#### Escribir datos en una managed table en el contenedor gold

In [0]:
#delete_dups("movie_gold_inc","results_country_prod_company","created_date",f"{v_file_date}")












In [0]:
#results_order_by_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold_inc.results_country_prod_company")

merge_condition = 'tgt.movie_id = src.movie_id and tgt.country_id = src.country_id and tgt.company_id = src.company_id and tgt.created_date = src.created_date'
merge_delta_lake(results_order_by_df,'movie_gold_inc','results_country_prod_company',merge_condition,'created_date')


















In [0]:
%sql
SELECT created_date, COUNT(1) 
FROM movie_gold_inc.results_country_prod_company
GROUP BY created_date